# 08 边界审查修复 — pytest 全量回归(超时跳过)

回归 `tests/` 全部 pytest 文件, 命令与规则同 `scripts/run_pytest_timeout_skip.py`:
- 文件级超时 900s / 用例级超时 120s(pytest-timeout thread 法)
- **超时整文件记 SKIPPED 不算失败**(约定: 外部依赖 PG 容器因 Docker 引擎故障不可用)
- 结论行: `ALL PASSED` / `HAS FAILURES`(SKIPPED 不触发失败)

因 4 个 PG 依赖文件每个要烧 ~300s 超时预算, 超过 run_nb 单 cell 600s 上限,
按文件分批执行, 内核内共享 `results` 聚合。

In [1]:
import subprocess
import sys
from pathlib import Path

PY = sys.executable
ROOT = Path.cwd().parent if Path.cwd().name == "tests_ipynb" else Path.cwd()
SCRIPT = str(ROOT / "scripts" / "run_pytest_timeout_skip.py")

FAST_FILES = [  # 无 PG 依赖/秒级完成
    "tests/test_boundary_high_backend.py",
    "tests/test_boundary_medium.py",
    "tests/test_boundary_low.py",
    "tests/test_engineering.py",
    "tests/test_p0_retrieval.py",
    "tests/test_tracing.py",
]
SLOW_FILES = [  # PG/子进程依赖, 可能吃满超时预算, 每文件独立 cell
    "tests/test_mcp.py",
    "tests/test_sessions_degrade.py",
    "tests/test_smoke.py",
    "tests/test_trace_db.py",
    "tests/test_trace_e2e.py",
]

results = []  # (文件批次名, 汇总行, 结论)

def run_batch(name, files):
    args = [PY, SCRIPT, *files]
    r = subprocess.run(args, capture_output=True, text=True, cwd=str(ROOT), timeout=580)
    out = (r.stdout or "") + (r.stderr or "")
    print(f"=== {name} (rc={r.returncode}) ===")
    print(out)
    lines = [l for l in out.splitlines() if l.strip()]
    summary = next((l for l in lines if l.startswith("# 汇总")), "")
    verdict = lines[-1] if lines else "HAS FAILURES"
    results.append((name, summary, verdict, r.returncode))
    return r.returncode

print("批量回归初始化完成 | 解释器:", PY)

批量回归初始化完成 | 解释器: F:\Anaconda_env\lawApp_langGraph\python.exe


In [2]:
run_batch('快批(6 文件, 无 PG 依赖)', FAST_FILES)

=== 快批(6 文件, 无 PG 依赖) (rc=0) ===
# pytest 超时跳过跑测: 6 个文件, 文件超时 900s / 用例超时 120s
[PASS] tests/test_boundary_high_backend.py: 6s -- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html | 24 passed, 2 warnings in 4.00s
[PASS] tests/test_boundary_medium.py: 8s -- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html | 27 passed, 2 warnings in 6.30s
[PASS] tests/test_boundary_low.py: 5s -- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html | 17 passed, 2 warnings in 3.54s
[PASS] tests/test_engineering.py: 2s -- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html | 4 passed, 1 warning in 0.76s
[PASS] tests/test_p0_retrieval.py: 1s ...                                                                      [100%] | 3 passed in 0.10s
[PASS] tests/test_tracing.py: 5s -- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html | 10 passed, 2 warnings in 3.14s

# 汇总: 6 PASS / 0 SKIPPED / 0 FAIL
ALL PASSED



0

In [3]:
run_batch('tests/test_mcp.py', ['tests/test_mcp.py'])

=== tests/test_mcp.py (rc=0) ===
# pytest 超时跳过跑测: 1 个文件, 文件超时 900s / 用例超时 120s
[PASS] tests/test_mcp.py: 82s -- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html | 5 passed, 1 warning in 79.37s (0:01:19)

# 汇总: 1 PASS / 0 SKIPPED / 0 FAIL
ALL PASSED



0

In [4]:
run_batch('tests/test_sessions_degrade.py', ['tests/test_sessions_degrade.py'])

=== tests/test_sessions_degrade.py (rc=0) ===
# pytest 超时跳过跑测: 1 个文件, 文件超时 900s / 用例超时 120s
[PASS] tests/test_sessions_degrade.py: 6s -- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html | 1 passed, 2 warnings in 4.13s

# 汇总: 1 PASS / 0 SKIPPED / 0 FAIL
ALL PASSED



0

In [5]:
run_batch('tests/test_smoke.py', ['tests/test_smoke.py'])

=== tests/test_smoke.py (rc=0) ===
# pytest 超时跳过跑测: 1 个文件, 文件超时 900s / 用例超时 120s
[PASS] tests/test_smoke.py: 7s -- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html | 11 passed, 2 warnings in 5.68s

# 汇总: 1 PASS / 0 SKIPPED / 0 FAIL
ALL PASSED



0

In [6]:
run_batch('tests/test_trace_db.py', ['tests/test_trace_db.py'])

=== tests/test_trace_db.py (rc=0) ===
# pytest 超时跳过跑测: 1 个文件, 文件超时 900s / 用例超时 120s
[PASS] tests/test_trace_db.py: 2s ...                                                                      [100%] | 3 passed in 0.70s

# 汇总: 1 PASS / 0 SKIPPED / 0 FAIL
ALL PASSED



0

In [7]:
run_batch('tests/test_trace_e2e.py', ['tests/test_trace_e2e.py'])

=== tests/test_trace_e2e.py (rc=0) ===
# pytest 超时跳过跑测: 1 个文件, 文件超时 900s / 用例超时 120s
[PASS] tests/test_trace_e2e.py: 7s -- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html | 1 passed, 2 warnings in 5.37s

# 汇总: 1 PASS / 0 SKIPPED / 0 FAIL
ALL PASSED



0

In [8]:
import re

print()
print("# ===== 边界修复回归汇总 =====")
total = {'PASS': 0, 'SKIPPED': 0, 'FAIL': 0}
for name, summary, verdict, rc in results:
    print(f'- {name}: {summary or verdict}')
    for key in ('PASS', 'SKIPPED', 'FAIL'):
        m = re.search('(0|[1-9][0-9]*) ' + key, summary)
        if m:
            total[key] += int(m.group(1))

fail_batches = [name for name, s, v, rc in results if v == 'HAS FAILURES' or rc not in (0, 1)]
print()
print(f"# 汇总: {len(results)} 批 | 用例 {total['PASS']} PASS / {total['SKIPPED']} SKIPPED / {total['FAIL']} FAIL")
print('ALL PASSED' if not fail_batches else 'HAS FAILURES: ' + str(fail_batches))


# ===== 边界修复回归汇总 =====
- 快批(6 文件, 无 PG 依赖): # 汇总: 6 PASS / 0 SKIPPED / 0 FAIL
- tests/test_mcp.py: # 汇总: 1 PASS / 0 SKIPPED / 0 FAIL
- tests/test_sessions_degrade.py: # 汇总: 1 PASS / 0 SKIPPED / 0 FAIL
- tests/test_smoke.py: # 汇总: 1 PASS / 0 SKIPPED / 0 FAIL
- tests/test_trace_db.py: # 汇总: 1 PASS / 0 SKIPPED / 0 FAIL
- tests/test_trace_e2e.py: # 汇总: 1 PASS / 0 SKIPPED / 0 FAIL

# 汇总: 6 批 | 用例 11 PASS / 0 SKIPPED / 0 FAIL
ALL PASSED
